# Chapter 7: Sharding
*From: Designing Data-Intensive Applications*

## TL;DR

- **Sharding (a.k.a. partitioning)** splits one logical dataset across many nodes so that a single machine's write throughput or storage is no longer the ceiling. Each record belongs to exactly one shard; each shard is usually also **replicated** to other nodes for fault tolerance.
- Two families of sharding schemes dominate: **key-range** (contiguous key intervals per shard, great for range scans, vulnerable to hot spots on monotonic keys like timestamps) and **hash-based** (uniform spread via a hash function, kills range queries on the partition key).
- Hash sharding shows up in three flavors — **hash mod N** (never use it; almost everything reshuffles when N changes), **fixed shard count** (hash(k) % S where S >> N), and **hash-range / consistent hashing** (number of shards grows with data).
- **Hot keys** (celebrities, viral posts) concentrate load on one shard even with a perfect hash. Relieve them by dedicating a shard, or by salting the key with a random suffix — at the cost of scatter-gather on reads.
- **Request routing** needs a consistent shard-to-node map. Most systems park this map in a consensus service (ZooKeeper, etcd, or a built-in Raft implementation); Riak's gossip-based alternative tolerates split brain. **Secondary indexes** force a choice: local (document-partitioned — cheap writes, scatter-gather reads) or global (term-partitioned — fan-out writes, one-shard reads).

## The Problem Sharding Solves

- **Write throughput and dataset size** outgrow a single machine. Replication scales reads but every replica still writes everything, so it does not solve this.
- **Multitenancy** in SaaS: each tenant gets its own shard (or small tenants are grouped), giving resource isolation, permission isolation, per-tenant backup, data residency, and GDPR-friendly deletion.
- **Cell-based architecture** extends sharding to the service tier — a fault in one cell stays in that cell.

**What sharding is NOT for:** fault tolerance (that's replication), cross-shard joins (that's distributed query execution), or read-heavy workloads that fit on one machine (use followers instead).

## Functional Requirements

- Route a read/write for any **partition key** to the correct shard in a single hop when possible.
- Support **range scans** on *some* keys (timestamps, IDs) at least within a single tenant.
- **Add and remove nodes** (rebalance) without full downtime and without moving vastly more data than necessary.
- Keep **secondary indexes** in sync with primary data, ideally with bounded staleness.
- Tolerate **hot keys** without wedging the whole cluster.

## Non-Functional Requirements

- **Scale out linearly**: 10x nodes ≈ 10x write throughput and 10x data capacity, as long as load is well-distributed.
- **Rebalance cost**: moving shards should not saturate the network or push an already-hot node over the edge.
- **Routing availability**: the shard-to-node map must survive the failure of any single coordinator node.
- **Target shard size**: roughly 1–10 GB per shard (HBase default split threshold is 10 GB) — small enough to move quickly, large enough to amortize metadata overhead.

## Back-of-Envelope Estimation

The quantitative core of the chapter is: how many shards do you need, how big is each, and how bad does a hot key get? We work all three below.

In [ ]:
# =====================================================================
# 1. How many shards for a 100 TB multi-tenant SaaS?
# =====================================================================

total_data_bytes = 100 * 1024**4          # 100 TB
target_shard_bytes = 10 * 1024**3         # 10 GB target (HBase default)
num_shards = total_data_bytes / target_shard_bytes

nodes = 50
shards_per_node = num_shards / nodes
data_per_node_gb = (total_data_bytes / nodes) / 1024**3

print(f"Total dataset:      {total_data_bytes / 1024**4:,.0f} TB")
print(f"Target shard size:  {target_shard_bytes / 1024**3:,.0f} GB")
print(f"Shards needed:      {num_shards:,.0f}")
print(f"Nodes:              {nodes}")
print(f"Shards per node:    {shards_per_node:,.0f}")
print(f"Data per node:      {data_per_node_gb:,.0f} GB")

# ---------------------------------------------------------------------
# 10,240 shards across 50 nodes = ~205 shards/node. Losing one node
# means its ~205 shards get reassigned — each shard's replica on
# another node is promoted and a new follower is built elsewhere.
# ---------------------------------------------------------------------

In [ ]:
# =====================================================================
# 2. Hash-mod-N catastrophe: how many keys move when you add one node?
# =====================================================================

def keys_moved_mod_n(n_before: int, n_after: int, total_keys: int) -> int:
    """
    With hash(k) % N, a key k moves iff (h % n_before) != (h % n_after).
    Over a uniform hash distribution, the fraction that stays is roughly
    1 / lcm(n_before, n_after) * min(n_before, n_after) — but empirically
    the vast majority move. We approximate with the intuition in the book.
    """
    from math import gcd
    lcm = n_before * n_after // gcd(n_before, n_after)
    # keys stay iff hash mod lcm lands in one of min(n_before, n_after) residues
    stays = min(n_before, n_after) / lcm
    return int(total_keys * (1 - stays))

total = 1_000_000_000
print(f"3 -> 4 nodes: {keys_moved_mod_n(3, 4, total):,} keys move  ({keys_moved_mod_n(3,4,total)/total*100:.1f}%)")
print(f"10 -> 11:     {keys_moved_mod_n(10, 11, total):,} keys move ({keys_moved_mod_n(10,11,total)/total*100:.1f}%)")
print(f"100 -> 101:   {keys_moved_mod_n(100, 101, total):,} keys move")

# ---------------------------------------------------------------------
# Even adding a single node reshuffles >90% of keys. This is why nobody
# uses naive hash mod N in production — they use a fixed shard count
# (hash % S where S >> N) or consistent hashing, both of which move
# only ~1/N of keys per node change.
# ---------------------------------------------------------------------

In [ ]:
# =====================================================================
# 3. Celebrity hot-key problem: how overloaded is one shard?
# =====================================================================

total_qps = 1_000_000                 # 1 M req/s across platform
num_shards = 1000
avg_qps_per_shard = total_qps / num_shards

# One celebrity accounts for 5% of all traffic, all on ONE shard
celebrity_fraction = 0.05
celebrity_qps = total_qps * celebrity_fraction
hot_shard_qps = celebrity_qps + (total_qps - celebrity_qps) / num_shards

print(f"Average shard QPS:   {avg_qps_per_shard:>10,.0f}")
print(f"Hot shard QPS:       {hot_shard_qps:>10,.0f}")
print(f"Hot shard is {hot_shard_qps / avg_qps_per_shard:,.1f}x the average")

# Mitigation: salt the celebrity's key across 100 sub-keys
salt_factor = 100
salted_hot_qps = celebrity_qps / salt_factor + (total_qps - celebrity_qps) / num_shards
print(f"After salting (x{salt_factor}): {salted_hot_qps:>10,.0f} QPS/shard (reads now scatter-gather across {salt_factor} keys)")

# ---------------------------------------------------------------------
# The celebrity shard sees ~50x the average load. Salting spreads the
# WRITE load but every read must now fan out to all 100 salted keys
# and merge. Worth it only for a handful of known hot keys; application
# must track which keys are salted.
# ---------------------------------------------------------------------

In [ ]:
# =====================================================================
# 4. Rebalance bandwidth budget: how long to move shards to a new node?
# =====================================================================

shards_to_move = 200                  # new node takes 200 shards
shard_size_gb = 10
data_to_move_gb = shards_to_move * shard_size_gb

# Throttle rebalance to avoid swamping live traffic
rebalance_bandwidth_mbps = 100        # 100 MB/s cap per donor node
seconds = (data_to_move_gb * 1024) / rebalance_bandwidth_mbps
print(f"Data to move:        {data_to_move_gb:,} GB")
print(f"Bandwidth cap:       {rebalance_bandwidth_mbps} MB/s")
print(f"Rebalance time:      {seconds / 60:,.1f} min  ({seconds/3600:.2f} h)")

# ---------------------------------------------------------------------
# ~5.5 hours to onboard a node — during this window the old assignment
# must still serve traffic, and the new node gradually takes over.
# This is why "automatic failure detection + automatic rebalance" is
# dangerous: a flaky node gets marked dead, triggers a multi-hour
# rebalance, loads up its peers, and they start looking flaky too.
# ---------------------------------------------------------------------

## High-Level Architecture

Sharding is always combined with replication. Each shard has one leader and N-1 followers, and each physical node is typically leader for some shards and follower for others.

```mermaid
graph TB
  subgraph Node1[Node 1]
    S1L[Shard 1<br/>LEADER]
    S2F[Shard 2<br/>follower]
    S3F[Shard 3<br/>follower]
  end
  subgraph Node2[Node 2]
    S1F[Shard 1<br/>follower]
    S2L[Shard 2<br/>LEADER]
    S4F[Shard 4<br/>follower]
  end
  subgraph Node3[Node 3]
    S3L[Shard 3<br/>LEADER]
    S4L[Shard 4<br/>LEADER]
    S1F2[Shard 1<br/>follower]
  end

  Client -->|write k in shard 1| S1L
  S1L -.replicate.-> S1F
  S1L -.replicate.-> S1F2
  Client -->|write k in shard 2| S2L
  S2L -.replicate.-> S2F
```

Losing Node 1 leaves Shard 1 leaderless — one of its followers is promoted, and a new follower is eventually rebuilt elsewhere. No data loss because the shard lives on 3 nodes.

## Deep Dive 1: Key-Range vs. Hash-Range Sharding

The choice of sharding key distribution is the most load-bearing decision in the whole system. Compare the two shapes:

```mermaid
graph LR
  subgraph KR[Key-Range Sharding]
    direction TB
    KR0["Shard 0<br/>A - F"]
    KR1["Shard 1<br/>G - M"]
    KR2["Shard 2<br/>N - S"]
    KR3["Shard 3<br/>T - Z"]
  end
  subgraph HR[Hash-Range Sharding]
    direction TB
    HR0["Shard 0<br/>hash 0x0000 - 0x3FFF"]
    HR1["Shard 1<br/>hash 0x4000 - 0x7FFF"]
    HR2["Shard 2<br/>hash 0x8000 - 0xBFFF"]
    HR3["Shard 3<br/>hash 0xC000 - 0xFFFF"]
  end

  Write1["write k='timestamp-2026-04-23'"] --> KR3
  Write1 -. "hashes to" .-> HR1
  Write2["write k='timestamp-2026-04-24'"] --> KR3
  Write2 -. "hashes to" .-> HR3
```

Sequential writes in key-range sharding pile onto the same shard (KR3); under hash-range, the hash spreads them. But a `WHERE time BETWEEN a AND b` range scan is a single-shard read under key-range — under hash-range it must scatter-gather.

**Practical escape hatch**: a compound partition key like `(sensor_id, timestamp)`. `sensor_id` distributes writes (many sensors write concurrently) while timestamp within a sensor stays sorted and range-scannable.

## Deep Dive 2: Rebalancing Strategies

Three practical schemes, each with different behavior when N changes:

**Hash mod N** (never use): hash(k) % N. Adding or removing one node moves ~(N-1)/N of all keys.

**Fixed shard count**: pick S >> N up front (say S=1000 for a 10-node cluster). Keys map to shards via hash(k) % S, which never changes. When nodes change, only the shard-to-node mapping changes; entire shards move as units. Used by Citus, Riak, Elasticsearch, Couchbase. Fails when dataset outgrows original S — an expensive resharding is required.

**Hash-range / dynamic splits**: hash space is divided into ranges; a shard whose size or load crosses a threshold splits into two. Used by YugabyteDB, DynamoDB, Cassandra (per-node vnodes), ScyllaDB. Number of shards grows with data.

```mermaid
graph TB
  subgraph Before[Before: 3 nodes, 12 shards]
    N1[Node 1<br/>shards 1,2,3,4]
    N2[Node 2<br/>shards 5,6,7,8]
    N3[Node 3<br/>shards 9,10,11,12]
  end
  subgraph After[After: add Node 4]
    N1b[Node 1<br/>shards 1,2,3]
    N2b[Node 2<br/>shards 5,6,7]
    N3b[Node 3<br/>shards 9,10,11]
    N4b[Node 4<br/>shards 4,8,12]
  end
  Before -->|"move 3 shards, not reshuffle"| After
```

Only 25% of shards (3 of 12) move — proportional to 1/new_N, not to N.

## Deep Dive 3: Request Routing and the Shard Map

Any client needs to know which node holds a given shard right now. Three ways to do it:

```mermaid
graph TB
  subgraph ZK[ZooKeeper / etcd / Raft]
    Map["Shard-to-Node Map<br/>shard_1 -> node_A<br/>shard_2 -> node_B<br/>..."]
  end

  Client1[Client 1<br/>shard-aware] -->|1. subscribe| Map
  Client1 -->|2. direct write| NodeB
  Client2[Client 2<br/>dumb] -->|1. any node| LB[Round-Robin LB]
  LB --> NodeA
  NodeA -->|3. forward if wrong shard| NodeB
  Client3[Client 3] -->|1. all requests| Router[Routing Tier<br/>mongos / proxy]
  Router -->|2. subscribe| Map
  Router -->|3. forward to owner| NodeB

  NodeA -.register.-> Map
  NodeB -.register.-> Map
  NodeC[Node C] -.register.-> Map
```

- **Any-node forwarding**: clients dumb, nodes do the work. One extra network hop on miss.
- **Routing tier** (MongoDB `mongos`, Vitess `vtgate`): a stateless proxy subscribing to the shard map. Clean separation but adds hop + deployment complexity.
- **Shard-aware client**: fastest, but every client SDK must implement and cache the map.

Riak uses **gossip** instead of consensus — weaker consistency, tolerates split brain, but fits its leaderless model where conflicts are resolved on read anyway.

## Deep Dive 4: Secondary Indexes — Local vs. Global

Secondary indexes ask a different question ("all red cars") than primary-key lookups ("car #742"). They don't align with the partition key.

```mermaid
graph TB
  subgraph Local[Local Secondary Index - document-partitioned]
    direction TB
    LS0["Shard 0<br/>car IDs 0-499<br/>local idx: color, make"]
    LS1["Shard 1<br/>car IDs 500-999<br/>local idx: color, make"]
    Read1["Query: color=red"] --> LS0
    Read1 --> LS1
    LS0 --> Merge[Scatter-Gather<br/>merge results]
    LS1 --> Merge
  end
  subgraph Global[Global Secondary Index - term-partitioned]
    direction TB
    GS0["Shard A<br/>color a-r<br/>term idx: 'red' -> IDs"]
    GS1["Shard B<br/>color s-z<br/>term idx: 'silver' -> IDs"]
    Write["Write: add red car #742"] --> GS0
    Write --> PrimaryShard[Primary shard holds the car record]
    ReadG["Query: color=red"] --> GS0
  end
```

- **Local**: write to the record's shard touches *only* that shard — cheap writes, but reads must hit every shard (tail-latency amplification). MongoDB, Cassandra, Elasticsearch, Riak, VoltDB.
- **Global**: a single record's write fans out to many index shards — expensive writes (often async, hence stale reads), but point reads on a term go to one shard. CockroachDB, TiDB, YugabyteDB, DynamoDB (both modes).

## Trade-offs and Alternatives

### Sharding scheme

| Scheme | Range Scans | Load Distribution | Rebalance Cost | When to Use |
|--------|-------------|-------------------|-----------------|-------------|
| Key-range | Efficient (single-shard) | Can hot-spot on monotonic keys | Split/merge shards | HBase, CockroachDB, MongoDB range mode; time-series with composite key |
| Hash mod N | No (shuffled) | Uniform | Catastrophic (~all keys move) | Never — educational only |
| Fixed shard count | Only within same partition key | Uniform | Cheap (move whole shards) | Citus, Riak, Elasticsearch, Couchbase when you can estimate scale up front |
| Hash-range / consistent hashing | Only within same partition key | Uniform with minor variance | Cheap and adaptive | Cassandra, ScyllaDB, DynamoDB, YugabyteDB when dataset size is unpredictable |

### Secondary index

| Approach | Write Cost | Read Cost | Consistency | Users |
|----------|-----------|-----------|-------------|-------|
| Local (document-partitioned) | O(1) — only local shard | O(N) shards — scatter-gather | Synchronous — index stays in sync trivially | MongoDB, Cassandra, Elasticsearch, Riak, VoltDB |
| Global (term-partitioned) | O(terms) — fan-out to each term's shard | O(1) shard per term | Usually async — reads can be stale | CockroachDB, TiDB, YugabyteDB, DynamoDB |

### Rebalancing strategy

| Strategy | Pros | Cons | When to Use |
|----------|------|------|-------------|
| Fully automatic (DynamoDB adaptive capacity) | Zero operator toil, autoscales | Can cascade on false failure detection; unpredictable cost | Cloud services, well-understood workloads |
| Suggested, admin-confirmed (Couchbase, Riak) | Automation + safety net | Still needs a human on-call | Mid-size teams that want a kill switch |
| Fully manual (Vitess) | Maximum predictability; can preemptively scale for Black Friday | Ops-heavy; reaction time slow | Very large / very critical systems with dedicated SRE teams |

## Key Takeaways

1. **Shard for volume or write throughput, not for reads.** Replication scales reads; sharding exists because one machine cannot hold the data or absorb the writes.
2. **Sharding is an irreversible commitment** to a partition key. Changing it later is an O(dataset) rewrite. Pick it knowing your dominant access pattern.
3. **Hash mod N is a trap.** Always use a fixed shard count (S >> N) or consistent hashing so rebalancing moves only 1/N of data per node change.
4. **Even a perfect hash cannot fix skewed load.** Hot keys (celebrities, viral events) must be salted, isolated on a dedicated shard, or handled with adaptive-capacity autoscaling.
5. **A consistent shard-to-node map is the single most important piece of cluster metadata.** Park it in ZooKeeper, etcd, or a built-in Raft — never in a single coordinator without a failover story.
6. **Secondary indexes force a fundamental trade.** Local indexes make writes cheap and reads scatter-gather; global indexes flip the cost. Pick based on your read/write ratio.
7. **Automatic rebalancing plus automatic failure detection can cause cascading failures.** A human-in-the-loop confirmation is cheap insurance at scale.

## See Also

- [[01-sharding-fundamentals-and-multitenancy]] — why shard at all, and why multitenancy is the unsung killer use case
- [[02-key-range-sharding]] — contiguous ranges, pre-splitting, hot spots on monotonic keys
- [[03-hash-based-sharding]] — hash mod N vs. fixed shard count vs. hash-range vs. consistent hashing
- [[04-skewed-workloads-and-hot-spots]] — the celebrity problem, key salting, adaptive capacity
- [[05-rebalancing-strategies]] — automatic vs. suggested vs. manual, and the cascading-failure trap
- [[06-request-routing]] — any-node forwarding, routing tier, shard-aware client; ZooKeeper/etcd/Raft vs. gossip
- [[07-sharding-and-secondary-indexes]] — local (document-partitioned) vs. global (term-partitioned) indexes